# Update Mapping Resources Notebook

This notebook automates the process of executing SPARQL queries and saving the results in a structured JSON format. The main steps include:

1. Reading SPARQL query files from the `resources/queries` directory.
2. Executing each query against the SPARQL endpoint: `https://publications.europa.eu/webapi/rdf/sparql`.
3. Formatting the query results as JSON with proper indentation.
4. Saving the formatted JSON files to the `resources/mapping_files` directory.

This notebook ensures the output directory exists and provides basic logging to track the execution of queries and the status of results.

In [ ]:
import os
from pathlib import Path

# Define paths
PROJECT_PATH = Path(os.getcwd()).resolve().parent
TED_SWS_PATH = PROJECT_PATH / "ted_sws"
queries_dir = TED_SWS_PATH / "resources" / "queries"
output_dir = TED_SWS_PATH / "resources" / "mapping_files"
endpoint_url = "https://publications.europa.eu/webapi/rdf/sparql"

JSON_IDENT = 2

In [ ]:
import requests
import json


# Ensure the output directory exists
output_dir.mkdir(parents=True, exist_ok=True)

# Iterate through all SPARQL query files in the queries directory
for query_file in queries_dir.glob("*.rq"):
    # Read the SPARQL query
    with query_file.open("r", encoding="utf-8") as file:
        sparql_query = file.read()

    # Prepare the request parameters
    params = {
        "default-graph-uri": "",
        "query": sparql_query,
        "format": "application/sparql-results+json",
        "timeout": 0,
        "debug": "on"
    }

    # Execute the query
    print(f"Executing query: {query_file.name}")
    response = requests.get(endpoint_url, params=params)
    print(f"Response status code for query {query_file.name}: {response.status_code}")

    if response.status_code == 200:
        # Save the result in the output directory
        output_file = output_dir / f"{query_file.stem}.json"
        with output_file.open("w", encoding="utf-8") as file:
            # Format the JSON response before saving
            json_data = response.json()
            json.dump(json_data, file, indent=JSON_IDENT, ensure_ascii=False)
        print(f"Saved formatted results for {query_file.name} to {output_file}")
    else:
        print(f"Failed to execute query {query_file.name}. HTTP Status Code: {response.status_code}")